## Using an Individualized Baseline Approach to Predict Reinjury Among MLB Batters 

In [1]:
## Install pybaseball ##

%pip install pybaseball


Note: you may need to restart the kernel to use updated packages.


In [2]:
## Install seaborn ##

%pip install seaborn

## Install openpyxl ##

%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
## Load Relevant Packages ##

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pybaseball as pb

## Read in MLB Injuries Raw Data for 2022 Season ##

url = "https://raw.githubusercontent.com/abrown9008/Sports-Analytics-and-Intelligence-Lab/main/MLB%20Oblique%20Injuries/MLB%20Injuries%20Raw%20Data.xlsx"
injuries22 = pd.read_excel(url, engine="openpyxl",sheet_name='2022')

print(injuries22.head())

              Name                                                URL Team  \
0     Tommy Kahnle  https://www.fangraphs.com/players/tommy-kahnle...  LAD   
1   Caleb Ferguson  https://www.fangraphs.com/players/caleb-fergus...  LAD   
2       Nick Mears  https://www.fangraphs.com/players/nick-mears/2...  PIT   
3  Kyle Funkhouser  https://www.fangraphs.com/players/kyle-funkhou...  DET   
4  Blake Cederlind  https://www.fangraphs.com/players/blake-cederl...  PIT   

  Pos Injury / Surgery Date            Injury / Surgery     Status  \
0  RP   2020-08-04 00:00:00          Tommy John surgery  Activated   
1  RP   2020-09-23 00:00:00          Tommy John surgery  Activated   
2  RP   2021-02-09 00:00:00  Arthroscopic elbow surgery  Activated   
3  RP               Mar '22           Strained shoulder  60-Day IL   
4  RP   2021-03-23 00:00:00          Tommy John surgery  60-Day IL   

  IL Retro Date Eligible to Return Return Date            Latest Update  \
0    2022-04-04         2022-04-14 

In [2]:
## Pivot Data such that each row is a unique player ##

stint_cols = {
    'IL Retro Date': 'IL_Retro',
    'Return Date': 'Activate',
    'Injury / Surgery': 'Injury',
}

df = injuries22.copy()
for c in ['IL Retro Date', 'Return Date']:
    df[c] = pd.to_datetime(df[c], errors='coerce')

# Stint 1 = earliest; rows with no IL date sort last
df = df.sort_values(['key_fangraphs', 'IL Retro Date'], na_position='last')
df['stint'] = df.groupby('key_fangraphs').cumcount() + 1

# Player-level attributes (first stint's value is kept if a player changed team)
players = df[['key_fangraphs', 'Name', 'Team', 'Pos', 'URL', 'sub_url']] \
    .drop_duplicates('key_fangraphs')

wide = (
    df.set_index(['key_fangraphs', 'stint'])[list(stint_cols)]
      .rename(columns=stint_cols)
      .unstack('stint')
)
wide.columns = [f'{col}_{n}' for col, n in wide.columns]

max_n = int(df['stint'].max())
ordered = [f'{c}_{n}' for n in range(1, max_n + 1) for c in stint_cols.values()]
wide = wide[ordered].reset_index()

injuries22_wide = players.merge(wide, on='key_fangraphs', how='left')

assert injuries22_wide['key_fangraphs'].is_unique
print(injuries22_wide.shape, df['stint'].value_counts().sort_index().to_dict())



(722, 24) {1: 722, 2: 210, 3: 42, 4: 5, 5: 1, 6: 1}


In [3]:
## Now let's subset to only non-pitchers and oblique injuries for the 2022 season ##

inj22_wide_batters_oblique = injuries22_wide[
    (injuries22_wide['Pos'] != 'P') &
    (injuries22_wide['Pos'] != 'SP') &
    (injuries22_wide['Pos'] != 'RP') &
    (injuries22_wide['Injury_1'].str.contains('oblique', case=False, na=False))
]

In [4]:
## Reusable: read a season sheet and pivot to one row per player ##

stint_cols = {
    'IL Retro Date': 'IL_Retro',
    'Return Date': 'Activate',
    'Injury / Surgery': 'Injury',
}
keep_cols = ['Name', 'URL', 'Team', 'Pos', 'IL Retro Date', 'Return Date',
             'Injury / Surgery', 'sub_url', 'key_fangraphs']

def build_wide(year):
    raw = pd.read_excel(url, engine="openpyxl", sheet_name=str(year))[keep_cols]

    # Recover missing keys from the FanGraphs URL (e.g. /players/name/19931/stats/batting)
    raw['key_fangraphs'] = (
        raw['key_fangraphs']
        .fillna(raw['URL'].str.extract(r'/players/[^/]+/([^/]+)/')[0])
        .astype(str)
    )

    df = raw.copy()
    for c in ['IL Retro Date', 'Return Date']:
        df[c] = pd.to_datetime(df[c], errors='coerce')

    df = df.sort_values(['key_fangraphs', 'IL Retro Date'], na_position='last')
    df['stint'] = df.groupby('key_fangraphs').cumcount() + 1

    players = df[['key_fangraphs', 'Name', 'Team', 'Pos', 'URL', 'sub_url']] \
        .drop_duplicates('key_fangraphs')

    wide = (
        df.set_index(['key_fangraphs', 'stint'])[list(stint_cols)]
          .rename(columns=stint_cols)
          .unstack('stint')
    )
    wide.columns = [f'{col}_{n}' for col, n in wide.columns]
    max_n = int(df['stint'].max())
    ordered = [f'{c}_{n}' for n in range(1, max_n + 1) for c in stint_cols.values()]
    wide = wide[ordered].reset_index()

    out = players.merge(wide, on='key_fangraphs', how='left')
    assert out['key_fangraphs'].is_unique
    assert out['key_fangraphs'].ne('nan').all(), f"{year}: unresolved keys"
    return out

## Build one wide dataframe per season ##

injuries_wide = {yr: build_wide(yr) for yr in [2021, 2022, 2023, 2024, 2025, 2026]}

injuries21_wide = injuries_wide[2021]
injuries23_wide = injuries_wide[2023]
injuries24_wide = injuries_wide[2024]
injuries25_wide = injuries_wide[2025]
injuries26_wide = injuries_wide[2026]

for yr, d in injuries_wide.items():
    print(yr, d.shape)

2021 (843, 18)
2022 (722, 24)
2023 (677, 18)
2024 (619, 18)
2025 (634, 18)
2026 (675, 18)


In [5]:
## Row bind and add season column ##

all_wide = pd.concat(
    [d.assign(Season=yr) for yr, d in injuries_wide.items()],
    ignore_index=True
)

In [6]:
## Subset all_wide to only non-pitchers and oblique injuries for all seasons ##

all_wide_batters_oblique = all_wide[
    (all_wide['Pos'] != 'P') &
    (all_wide['Pos'] != 'SP') &
    (all_wide['Pos'] != 'RP') &
    (all_wide['Injury_1'].str.contains('oblique', case=False, na=False))
]

In [7]:
## Remove players with NA in Active_1 column ##

all_wide_batters_oblique = all_wide_batters_oblique[
    all_wide_batters_oblique['Activate_1'].notna()
]

In [8]:
## How many players whose first injury was an oblique injury experienced a second oblique injury? ##

print("Total Number of Players with Oblique Injuries:", all_wide_batters_oblique.shape[0])
print("Number of Players with a Second Injury:", all_wide_batters_oblique['Injury_2'].notna().sum())
print("Number of Players with a Second Oblique Injury:", all_wide_batters_oblique['Injury_2'].str.contains('oblique', case=False, na=False).sum())

Total Number of Players with Oblique Injuries: 157
Number of Players with a Second Injury: 43
Number of Players with a Second Oblique Injury: 7


In [9]:
## Print the types of second injuries for players whose first injury was an oblique injury ##

print("Types of Second Injuries for Players with First Oblique Injury:")
print(all_wide_batters_oblique['Injury_2'].value_counts(dropna=False))

Types of Second Injuries for Players with First Oblique Injury:
Injury_2
NaN                                       114
Strained hamstring                          8
Strained oblique                            7
Strained quad                               3
Sprained ankle                              2
Knee inflammation                           1
Sprained thumb                              1
Shoulder subluxation (left)                 1
Fractured finger (left pinky)               1
Knee discomfort (patellar subluxation)      1
Sprained knee                               1
Forearm surgery (fracture)                  1
Lower back inflammation                     1
Plantar fascitis                            1
Fractured hand (right pinkie)               1
Strained lower back                         1
Back inflammation                           1
Fractured thumb (left)                      1
Ankle/foot tendinitis                       1
Back spasms                                 1
Spraine

In [10]:
## Okay so we're going to group secondary injuries into categories. Let's define a function to do that. ##

# ---------------------------------------------------------------------------
# 1. Exact-match mapping for the injury labels currently in the data.
#    Editing this dict is the place to record judgment calls (e.g., hamate).
# ---------------------------------------------------------------------------
EXACT_MAP = {
    "Strained oblique": "oblique_recurrence",
 
    "Strained hamstring": "lower_body_strain",
    "Strained quad": "lower_body_strain",
    "Strained calf": "lower_body_strain",
    "Strained hip flexor": "lower_body_strain",
 
    "Strained lower back": "lower_back",
    "Lower back inflammation": "lower_back",
    "Back inflammation": "lower_back",
    "Back spasms": "lower_back",
 
    "Knee inflammation": "knee_nontraumatic",
    "Knee discomfort (patellar subluxation)": "knee_nontraumatic",
    "Sprained knee": "knee_nontraumatic",
    "Arthroscopic knee surgery": "knee_nontraumatic",
 
    "Plantar fascitis": "other_overuse",
    "Ankle/foot tendinitis": "other_overuse",
    "Hand inflammation": "other_overuse",
 
    "Sprained ankle": "ankle_sprain",
 
    "Fractured finger (left pinky)": "traumatic",
    "Fractured hand (right pinkie)": "traumatic",
    "Fractured thumb (left)": "traumatic",
    "Forearm surgery (fracture)": "traumatic",
    "Hamate surgery (left hand)": "traumatic",  # judgment call: possibly swing-related
    "Sprained thumb": "traumatic",
    "Sprained hand (left)": "traumatic",
    "Fractured knee": "traumatic",
    "Knee bone bruise": "traumatic",
    "Shoulder subluxation (left)": "traumatic",
}
 
# ---------------------------------------------------------------------------
# 2. Keyword fallback for labels not in EXACT_MAP (e.g., when you add seasons).
#    Order matters: the first matching rule wins. Anything matched here is
#    flagged so you can review it and add it to EXACT_MAP.
# ---------------------------------------------------------------------------
KEYWORD_RULES = [
    (["fracture", "hamate", "bruise", "contusion", "hit by pitch", "concussion",
      "subluxation (left)", "subluxation (right)", "dislocat"], "traumatic"),
    (["oblique"], "oblique_recurrence"),
    (["back", "lumbar", "spasm"], "lower_back"),
    (["hamstring", "quad", "calf", "hip flexor", "groin", "adductor"], "lower_body_strain"),
    (["knee", "patella", "meniscus"], "knee_nontraumatic"),
    (["ankle sprain", "sprained ankle"], "ankle_sprain"),
    (["sprained hand", "sprained thumb", "sprained finger", "sprained wrist"], "traumatic"),
    (["tendinitis", "tendinopathy", "fasci", "inflammation", "bursitis"], "other_overuse"),
]
 
PRIMARY_GROUPS = {"oblique_recurrence", "lower_body_strain", "lower_back"}
SENSITIVITY_GROUPS = PRIMARY_GROUPS | {"knee_nontraumatic", "other_overuse", "ankle_sprain"}
NEGATIVE_CONTROL_GROUPS = {"traumatic"}
 
 
def classify_injury(label):
    """Return (group, match_method) for a single injury label."""
    if pd.isna(label) or str(label).strip() == "":
        return "none", "none"
 
    label = str(label).strip()
    if label in EXACT_MAP:
        return EXACT_MAP[label], "exact"
 
    lowered = label.lower()
    for keywords, group in KEYWORD_RULES:
        if any(k in lowered for k in keywords):
            return group, "keyword"
 
    return "unclassified", "unmatched"
 
 
def add_injury_groups(df, injury_col="Injury_2"):
    """Add group, outcome flags, and competing-risks event codes to df."""
    out = df.copy()
    results = out[injury_col].apply(classify_injury)
    out["injury_group"] = results.str[0]
    out["match_method"] = results.str[1]
 
    has_injury = out["injury_group"] != "none"
    out["primary_outcome"] = out["injury_group"].isin(PRIMARY_GROUPS).astype(int)
    out["sensitivity_outcome"] = out["injury_group"].isin(SENSITIVITY_GROUPS).astype(int)
    out["negative_control"] = out["injury_group"].isin(NEGATIVE_CONTROL_GROUPS).astype(int)
 
    # Competing-risks event codes: 0 = censored, 1 = event of interest, 2 = competing
    out["event_primary"] = np.select(
        [out["primary_outcome"] == 1, has_injury],
        [1, 2],
        default=0,
    )
    out["event_sensitivity"] = np.select(
        [out["sensitivity_outcome"] == 1, has_injury],
        [1, 2],
        default=0,
    )
 
    # Warn about anything that wasn't an exact match
    review = out.loc[out["match_method"].isin(["keyword", "unmatched"]), injury_col]
    if not review.empty:
        print("WARNING: these labels were not exact matches; review and add to EXACT_MAP:")
        for label, method in (
            out.loc[review.index, [injury_col, "match_method"]]
            .drop_duplicates()
            .itertuples(index=False)
        ):
            print(f"  [{method}] {label!r} -> {classify_injury(label)[0]}")
        print()
 
    return out
 
 
def summarize(grouped):
    """Print group counts and outcome totals."""
    print("Counts by injury group:")
    print(grouped["injury_group"].value_counts().to_string())
    print()
    injured = grouped["injury_group"] != "none"
    print(f"Players with a second injury: {injured.sum()} of {len(grouped)}")
    print(f"Primary outcome events:      {grouped['primary_outcome'].sum()}")
    print(f"Sensitivity outcome events:  {grouped['sensitivity_outcome'].sum()}")
    print(f"Negative control events:     {grouped['negative_control'].sum()}")
 
 
if __name__ == "__main__":
    # Example using the counts from the current data. Replace this block with
    # your own DataFrame, e.g.: df = pd.read_csv("oblique_players.csv")
    counts = {
        np.nan: 114,
        "Strained hamstring": 8, "Strained oblique": 7, "Strained quad": 3,
        "Sprained ankle": 2, "Knee inflammation": 1, "Sprained thumb": 1,
        "Shoulder subluxation (left)": 1, "Fractured finger (left pinky)": 1,
        "Knee discomfort (patellar subluxation)": 1, "Sprained knee": 1,
        "Forearm surgery (fracture)": 1, "Lower back inflammation": 1,
        "Plantar fascitis": 1, "Fractured hand (right pinkie)": 1,
        "Strained lower back": 1, "Back inflammation": 1,
        "Fractured thumb (left)": 1, "Ankle/foot tendinitis": 1,
        "Back spasms": 1, "Sprained hand (left)": 1, "Strained calf": 1,
        "Hand inflammation": 1, "Hamate surgery (left hand)": 1,
        "Arthroscopic knee surgery": 1, "Fractured knee": 1,
        "Strained hip flexor": 1, "Knee bone bruise": 1,
    }
    df = pd.DataFrame({"Injury_2": [k for k, n in counts.items() for _ in range(n)]})
 
    grouped = add_injury_groups(df, injury_col="Injury_2")
    summarize(grouped)

Counts by injury group:
injury_group
none                  114
lower_body_strain      13
traumatic              10
oblique_recurrence      7
knee_nontraumatic       4
lower_back              4
other_overuse           3
ankle_sprain            2

Players with a second injury: 43 of 157
Primary outcome events:      24
Sensitivity outcome events:  33
Negative control events:     10


In [11]:
grouped = add_injury_groups(all_wide_batters_oblique, injury_col="Injury_2")
summarize(grouped)

Counts by injury group:
injury_group
none                  114
lower_body_strain      13
traumatic              10
oblique_recurrence      7
knee_nontraumatic       4
lower_back              4
other_overuse           3
ankle_sprain            2

Players with a second injury: 43 of 157
Primary outcome events:      24
Sensitivity outcome events:  33
Negative control events:     10
